In [ ]:
import pandas as pd

df = pd.read_csv("data/readings.csv")

In [2]:
def load_sessions(paths): # list of file paths
  frames = []
  for p in paths:
        frames.append(pd.read_csv(p))
  return pd.concat(frames, keys=paths)

In [3]:
combined = load_sessions(["data/readings.csv"])
combined.shape
combined.index[:3]

MultiIndex([], )

In [8]:
def clean(df):
  before = len(df)
  sensors = df[["s0","s1","s2","s3","s4","s5"]]
  ok = ((sensors >= 0) & (sensors <= 4095)).all(axis=1)
  print("dropped:", before - len(df[ok]), "of", before)
  return df[ok]



In [9]:
cleaned = clean(combined)
print(cleaned.shape)

dropped: 0 of 6000
(6000, 8)


In [10]:
import numpy as np
values = cleaned[["s0","s1","s2","s3","s4","s5"]].to_numpy()
print(values.shape)
first = values[0:100, :]
print(first.shape)

(6000, 6)
(100, 6)


In [11]:
def window(values, length, stride):
  n = (len(values) - length) // stride + 1
  out = []
  for k in range(n):
        start = k * stride
        out.append(values[start : start + length, :])
  return np.stack(out)

w = window(values, 100, 50)
print(w.shape)

(119, 100, 6)


In [12]:
sensors = cleaned[["s0","s1","s2","s3","s4","s5"]]

summary = pd.DataFrame({
    "mean": sensors.mean(),
    "peak": sensors.max(),
    "active_frac": (sensors > 100).mean(),
})

print(summary)

          mean  peak  active_frac
s0  469.040833  1951     0.399000
s1  492.813167  1557     0.533167
s2  797.923833  2144     0.600000
s3  822.982167  2251     0.600000
s4  570.437167  1843     0.515667
s5  388.058333  1655     0.382667


In [13]:
toy = cleaned.head(20)
toy.to_csv("toy.csv", index=False)

In [14]:
t = load_sessions(["toy.csv"])
t = clean(t)
tv = t[["s0","s1","s2","s3","s4","s5"]].to_numpy()
tw = window(tv, 5, 5)
print(tw.shape)

dropped: 0 of 20
(4, 5, 6)
